# Injury to delta table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, ArrayType, BooleanType
from pyspark.sql.window import Window
from delta.tables import DeltaTable

ACCOUNT    = "footballmanagerli"
SRC        = f"abfss://alsource@{ACCOUNT}.dfs.core.windows.net/injury/"
DEST       = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/injury/injury_news"
CHECKPOINT = f"abfss://aldestination@{ACCOUNT}.dfs.core.windows.net/_checkpoints/injury_news"
TABLE      = "nfl.injury_news"

In [0]:
item_schema = StructType([
    StructField("id",               LongType()),
    StructField("created",          StringType()),
    StructField("created_formated", StringType()),
    StructField("author",           StringType()),
    StructField("player_id",        LongType()),
    StructField("team_id",          StringType()),
    StructField("title",            StringType()),
    StructField("sport_id",         StringType()),
    StructField("categories",       ArrayType(StringType())),
    StructField("link",             StringType()),
    StructField("desc",             StringType()),
    StructField("impact",           StringType()),
])

file_schema = StructType([
    StructField("sport",              StringType()),
    StructField("title",              StringType()),
    StructField("description",        StringType()),
    StructField("count",              LongType()),
    StructField("items",              ArrayType(item_schema)),
    StructField("limit",              LongType()),
    StructField("public_api_limited", BooleanType()),
    StructField("tier",               StringType()),
])

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS nfl")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {TABLE} (
    id           BIGINT    COMMENT 'FantasyPros news ID (primary key)',
    player_id    BIGINT,
    team_id      STRING,
    title        STRING,
    description  STRING,
    impact       STRING,
    author       STRING,
    categories   ARRAY<STRING>,
    link         STRING,
    sport_id     STRING,
    created_at   TIMESTAMP COMMENT 'When FantasyPros published it (UTC)',
    snapshot_at  TIMESTAMP COMMENT 'When the snapshot file containing this version was written',
    source_file  STRING,
    ingested_at  TIMESTAMP
)
USING DELTA
LOCATION '{DEST}'
""")

In [0]:
raw = (spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("multiLine", "true")
    .schema(file_schema)
    .load(SRC)
    .select(
        "items",
        F.col("_metadata.file_path").alias("source_file"),
        F.col("_metadata.file_modification_time").alias("snapshot_at"),
    ))

articles = (raw
    .select(F.explode("items").alias("a"), "source_file", "snapshot_at")
    .select(
        F.col("a.id").alias("id"),
        F.col("a.player_id").alias("player_id"),
        F.col("a.team_id").alias("team_id"),
        F.trim("a.title").alias("title"),
        F.trim("a.desc").alias("description"),
        F.trim("a.impact").alias("impact"),
        F.col("a.author").alias("author"),
        F.col("a.categories").alias("categories"),
        F.col("a.link").alias("link"),
        F.col("a.sport_id").alias("sport_id"),
        F.to_timestamp("a.created", "yyyy-MM-dd HH:mm:ss").alias("created_at"),
        "snapshot_at",
        "source_file",
        F.current_timestamp().alias("ingested_at"),
    ))

In [ ]:
def upsert_batch(batch_df, batch_id):
    """Upsert with automatic schema evolution: add missing columns if source has new fields."""
    
    # 1. Dedupe within the batch: the same article appears in many snapshots, so keep the newest
    w = Window.partitionBy("id").orderBy(F.col("snapshot_at").desc())
    latest = (batch_df
        .withColumn("_rn", F.row_number().over(w))
        .filter("_rn = 1")
        .drop("_rn"))
    
    # 2. Auto-add missing columns: if target table has columns not in batch, add them with NULLs
    #    This prevents merge failures when table schema differs from source data
    target_table = DeltaTable.forName(spark, TABLE)
    target_schema = {field.name: field.dataType for field in target_table.toDF().schema}
    batch_columns = {field.name for field in latest.schema}
    
    for col_name, col_type in target_schema.items():
        if col_name not in batch_columns:
            latest = latest.withColumn(col_name, F.lit(None).cast(col_type))
    
    # 3. Merge into the table: insert new articles, update only if this snapshot is newer
    (target_table.alias("t")
        .merge(latest.alias("s"), "t.id = s.id")
        .whenMatchedUpdateAll(condition="s.snapshot_at > t.snapshot_at")
        .whenNotMatchedInsertAll()
        .execute())

(articles.writeStream
    .foreachBatch(upsert_batch)
    .option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)   # process all new files, then stop
    .start()
    .awaitTermination())

## Schema Evolution

This notebook automatically handles schema evolution: if new columns appear in the source JSON, they are added to the Delta table with NULL values for existing rows. This prevents merge failures when schemas differ.

**How it works:**
1. Inside `foreachBatch`, before merge, we get the target table's schema
2. For any columns in the target that aren't in the batch, we add them with `F.lit(None).cast(type)`
3. Merge then succeeds even if source and target schemas differ

**Alternative (using library helper):**
```python
from lib.schema_evolution import auto_add_columns_to_batch

def upsert_batch(batch_df, batch_id):
    # ... dedupe logic ...
    latest = auto_add_columns_to_batch(latest, TABLE, spark)
    # ... merge ...
```

In [0]:
%sql
-- Should equal the number of unique articles
SELECT COUNT(*) AS rows, COUNT(DISTINCT id) AS unique_ids FROM nfl.injury_news;

-- Latest news
SELECT created_at, team_id, title, impact
FROM nfl.injury_news
ORDER BY created_at DESC
LIMIT 20;

-- One version per run
DESCRIBE HISTORY nfl.injury_news;

In [0]:
%sql
SELECT id, created_at, team_id, title, impact
FROM nfl.injury_news
ORDER BY created_at DESC
LIMIT 100;


In [0]:
%sql
SELECT current_catalog(), current_schema();